# QoolQit Exercises — Module 4 (Capstone)
## Putting it all together: solving a QUBO problem

Time to assemble everything from Modules 1–3 into a complete application:
solving a **QUBO** (Quadratic Unconstrained Binary Optimization) problem on a
neutral-atom quantum computer. Every step of the pipeline is one exercise —
each uses a tool you already practiced:

```
QUBO matrix ──▶ classical baseline (NumPy)
     │
     ▶ DataGraph.from_matrix          (Module 1)
     ▶ InteractionEmbedder            (Module 1)
     ▶ Register.from_graph            (Module 2)
     ▶ annealing Drive                (Module 2)
     ▶ QuantumProgram + compile_to    (Modules 2–3)
     ▶ LocalEmulator + histogram      (Module 3)
```


> **How to use this notebook.** 
> - Cells marked **✏️ Exercise** contain gaps
> indicated by `...` or `# TODO` — replace them with working code following
> the instructions. 
> - Cells marked **✅ Check** verify your answer: run them
> after completing the exercise. Everything else is provided and runs as-is.
> A separate **solution notebook** will be published.
>
> **API note:** we use qoolqit version 1.4

## 1. The problem

A QUBO instance on $N$ variables is a symmetric $N\times N$ matrix $Q$.
Solving it means finding the bitstring $z \in \{0,1\}^N$ minimizing

$$
f(z) = z^TQz= \sum_i Q_{ii} z_i + \sum_{i<j} Q_{ij} z_i z_j .
$$

Diagonal entries are *linear* costs (paid when $z_i = 1$); off-diagonal
entries are *pairwise* costs (paid when both bits are 1). Our instance:

$$ Q= \begin{pmatrix}
-10.0 & 0.52870372 & 0.52870372 & 0 & 0 \\
0.52870372 & -10.0 & 21.0 & 5.61708608 & 0 \\
0.52870372 & 21.0 & -10.0 & 0 & 5.61708608 \\
0 & 5.61708608 & 0 & -10.0 & 0 \\
0 & 0 & 5.61708608 & 0 & -10.0
\end{pmatrix} $$

**Why atoms can solve this natively.** In the Rydberg Hamiltonian with the
drive off ($\Omega = 0$),

$$
H = - \sum_i \delta_i \hat{n}_i + \sum_{i<j} \tilde{J}_{ij} \hat{n}_i \hat{n}_j ,
$$

which is *exactly* $f(z)$ under the dictionary
$Q_{ii} \leftrightarrow -\delta_i$, $Q_{ij} \leftrightarrow \tilde J_{ij}$,
$z_i = \hat n_i$. The **annealing protocol** exploits the adiabatic theorem:
start in the ground state $|0\rangle^{\otimes N}$ of an easy Hamiltonian,
evolve *slowly* while morphing it into the QUBO Hamiltonian, and the system
ends in the QUBO's ground state — the optimal bitstring — which we read out
by measuring.

### ✏️ Exercise 4.1 — Classical baseline by brute force

With $N=5$ there are only $2^5 = 32$ candidates, so we can compute the exact
optimum classically as ground truth (this scales exponentially — the whole
point of a quantum approach for large $N$!).

1. Enumerate all bitstrings (💡 `np.binary_repr(i, width)`).
2. Compute the cost of each candidate as $z^T\,\mathrm{triu}(Q)\,z$
   (💡 `np.triu` counts each pair once, matching $\sum_{i<j}$).
3. Sort by cost and save the **two best** bitstrings in
   `first_two_best_solutions`, printing them with their costs.

In [ ]:
import numpy as np

Q = np.array(
    [
        [-10.0, 0.52870372, 0.52870372, 0, 0],
        [0.52870372, -10.0, 21.0, 5.61708608, 0],
        [0.52870372, 21.0, -10.0, 0, 5.61708608],
        [0, 5.61708608, 0, -10.0, 0],
        [0, 0, 5.61708608, 0, -10.0],
    ]
)

# TODO: enumerate all 2**N candidate bitstrings (as strings)
solution_candidates = np.array([... for i in range(...)])

# TODO: costs z^T triu(Q) z
solution_candidates_list = np.array([... for b in solution_candidates])
costs = np.array([... for z in solution_candidates_list])

# TODO: sort and keep the two best
idx_sort = ...
sorted_costs = ...
sorted_solutions = ...

print("Two best minimizers: ", sorted_solutions[:2])
print("Respective costs: ", sorted_costs[:2])
first_two_best_solutions = sorted_solutions[:2]

In [ ]:
# ✅ Check — the optimum is a degenerate pair
assert set(first_two_best_solutions) == {"11011", "10111"}
assert np.isclose(sorted_costs[0], sorted_costs[1])
print("Classical optimum found:", first_two_best_solutions)
# (The degeneracy reflects a symmetry of Q: swapping variables 1<->2 and
#  3<->4 simultaneously leaves Q invariant and maps one optimum to the other.)

### ✏️ Exercise 4.2 — Load and embed the problem *(Module 1 tools)*

1. Since the QUBO is **scale invariant**, normalize it: `Q = Q / Q.max()`
   (this matches the embedder's convention $\max \tilde J = 1$ and
   simplifies the drive design).
2. *(Optional but instructive)* Build `DataGraph.from_matrix(Q.copy())` and
   draw it — the QUBO *is* a weighted graph.
3. Embed the matrix with an `InteractionEmbedder` into `embedded_graph`,
   draw it, and print each realized interaction next to its target `Q[i, j]`.
   The large couplings should match closely; exact zeros can only be
   approximated (atoms at finite distance always interact a little).

In [ ]:
from qoolqit import DataGraph

# TODO 1: normalize the QUBO (scale invariance)
Q = ...

# TODO 2 (optional): the QUBO as a weighted graph
graph = DataGraph.from_matrix(Q.copy())
graph.draw()

# TODO 3: embed the interaction matrix
embedded_graph = ...
embedded_graph.draw()

for (i, j), J in embedded_graph.interactions().items():
    print(f"pair ({i},{j}):  J = {J:.4f}   target Q = {Q[i, j]:.4f}")

### ✏️ Exercise 4.3 — Register and annealing Drive *(Module 2 tools)*

1. Build the register directly from the embedded graph:
   `Register.from_graph(embedded_graph)`.
2. Choose the annealing parameters:
   - `omega`: the **median** of the strictly positive entries of `Q`
     (a good handwavy value for the peak amplitude);
   - `delta_i = -1.0` (initial detuning: with $\Omega=0$ and $\delta<0$,
     $|0\rangle^{\otimes N}$ is the unique ground state);
   - `delta_f = -np.diag(Q)[0]` (final detuning matching the QUBO diagonal
     under $Q_{ii} \leftrightarrow -\delta_i$; all diagonal entries are
     equal here).
3. Build the schedule with `T = 40` (safely adiabatic, $\tilde t \gg 1$):
   a trapezoidal `PiecewiseLinearWaveform` amplitude
   ($0 \to \omega \to \omega \to 0$ over $[T/4, T/2, T/4]$ — exactly
   Exercise 2.2!) and a `RampWaveform` detuning from `delta_i` to `delta_f`.
4. Assemble the `Drive` and `draw()` it: check the boundary conditions of the
   annealing protocol at $t=0$ and $t=T$.

In [ ]:
# TODO 1: the register from the embedded graph
register = ...

# TODO 2: annealing parameters
omega = ...
delta_i = ...
delta_f = ...

# TODO 3: annealing schedule
T = 40
wf_amp = ...
wf_det = ...

# TODO 4: the drive
drive = ...
drive.draw()

### ✏️ Exercise 4.4 — Program, compilation and execution *(Module 3 tools)*

1. Assemble the `QuantumProgram` and compile it to an `AnalogDevice()`.
2. Run it on a `LocalEmulator` and store the measured
   `results.final_bitstrings` in `counts`.
3. Print the five most common bitstrings (`counts.most_common(5)`).

In [ ]:
# TODO: assemble and compile
program = ...
program.compile_to(device=..., profile="max_energy")

# TODO: run and collect counts
emulator = ...
counts = ...

print(counts.most_common(5))

### Plotting helper (provided)

Histogram of the sampled bitstrings; the classical optima from Exercise 4.1
are highlighted in **green**. Run as-is.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt


def plot_distribution(counter, solutions, bins=10):
    """Histogram of sampled bitstrings; known optimal `solutions` in green."""
    counter = Counter(counter)
    counter = dict(counter.most_common(bins))
    color = [
        "tab:green" if key in solutions.tolist() else "tab:blue" for key in counter
    ]
    _, ax = plt.subplots()
    ax.set_xlabel("Bitstrings")
    ax.set_ylabel("Counts")
    ax.bar(
        range(len(counter)), counter.values(), color=color, tick_label=counter.keys()
    )

### ✏️ Exercise 4.5 — Analyze… and improve!

1. Plot the distribution with the optima highlighted. **Look carefully**: the
   green bars are high, but is the *top* bar green? With this schedule the
   evolution is not adiabatic enough, and a suboptimal bitstring can win.
2. Now use the trick from the tutorial: recompile with
   `device_max_duration_ratio=1`, stretching the schedule to the device's
   **maximum allowed duration** (slower ⇒ more adiabatic). Re-run and plot
   again. The optima should now dominate clearly.

In [ ]:
# TODO 1: plot the first result
plot_distribution(..., ...)

In [ ]:
# TODO 2: recompile stretched to the device's maximum duration, re-run, re-plot
program.compile_to(device=..., profile="max_energy", device_max_duration_ratio=...)
counts_slow = ...
plot_distribution(..., ...)

In [ ]:
# ✅ Check — the two most sampled bitstrings are the classical optima
top2 = {b for b, _ in counts_slow.most_common(2)}
assert top2 == set(first_two_best_solutions), f"Top-2 sampled {top2} != optima"
print("🎉 QUBO solved: the classical optima are the most sampled bitstrings!")

## Going further

**Ideas to explore**
- Generate your own QUBO from a geometry (place atoms, compute $1/r^6$
  couplings, add a diagonal) and check the pipeline solves it.
- Shorten `T` and watch the solution quality degrade — quantify adiabaticity.
- Trigger the two classic `CompilationError`s from Module 3 with this
  register (amplitude too large; register too big).

## 🎓 Congratulations!

You built a complete neutral-atom application from first principles:
**graphs → embedding → register → drive → program → compilation → execution
→ verified quantum solution.** Every tool you used generalizes far beyond
QUBOs — happy experimenting with QoolQit!